# DatasetAgent Pipeline

Este notebook demuestra cómo ejecutar el pipeline de **DatasetAgent**, un sistema autónomo de descubrimiento y curación de conjuntos de datos científicos.

## ¿Qué es DatasetAgent?

DatasetAgent es un agente autónomo basado en grafos de estado diseñado para investigar, descubrir y organizar conjuntos de datos científicos de manera automatizada. En lugar de buscar manualmente en bases de datos, repositorios y literatura académica, este sistema:

1. **Genera consultas inteligentes** usando modelos de lenguaje (LLMs)
2. **Navega fuentes web** de forma autónoma para extraer metadatos
3. **Detecta duplicados** usando embeddings semánticos (similitud vectorial)
4. **Construye un grafo de conocimiento** conectando búsquedas, fuentes, observaciones y datasets
5. **Genera visualizaciones interactivas** para explorar las relaciones descubiertas

## ¿Por qué estas herramientas son necesarias en la investigación?

### El problema del descubrimiento de datasets

En investigación científica, especialmente en campos como la biología celular o la inmunología, los conjuntos de datos etiquetados son fundamentales para entrenar modelos de machine learning. Sin embargo:

- **Los datasets están dispersos**: publicados en repositorios, artículos, GitHub, sitios gubernamentales, etc.
- **No existe un índice centralizado**: cada fuente tiene su propio formato y estructura
- **La duplicación es común**: un mismo dataset puede aparecer en múltiples lugares con descripciones diferentes
- **La curación manual es lenta**: un investigador puede tardar semanas en encontrar y verificar todos los datasets relevantes

### Cómo DatasetAgent resuelve esto

| Herramienta | Problema que resuelve | Beneficio para la investigación |
|---|---|---|
| **Tavily Search** | Búsqueda web limitada de motores tradicionales | Encuentra fuentes que Google Scholar no indexa (repositorios institucionales, portales de datos gubernamentales) |
| **Wikidata SPARQL** | Falta de ontología estructural | Entiende las jerarquías biológicas (ej: neutrófilo → granulocito → leucocito) para guiar la búsqueda |
| **Embeddings semánticos** | Detección de duplicados que la coincidencia de texto no capta | Reconoce que "Dataset de imágenes de células sanguíneas" y "Blood Cell Image Collection" son el mismo recurso |
| **Grafo de estado (LangGraph)** | Coordinación de múltiples agentes autónomos | Orquesta el ciclo completo: descubrir → extraer → deduplicar → resumir |
| **Base de datos sqlite-vec** | Almacenamiento y búsqueda vectorial integrada | Mantiene un registro persistente con capacidad de búsqueda por similitud semántica |
| **Visualización interactiva** | Comprensión de relaciones complejas | Permite al investigador explorar el grafo de conocimiento y hacer clic en nodos para ver metadatos |

## Prerrequisitos

- Servidor vLLM corriendo con un modelo compatible (Llama, Qwen, etc.)
- Claves API configuradas en `.env` (ej: `TAVILY_API_KEY`)
- Dependencias instaladas via `uv sync`

## Uso

1. Inicia tu servidor LLM: `bash scripts/llama_serve.sh start`
2. Ejecuta este notebook desde el directorio raíz del proyecto
3. El pipeline esperará automáticamente a que el servidor esté listo

In [ ]:
import os
import sys
import yaml

# Asegurar que la raíz del proyecto está en el path
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Cargar variables de entorno desde .env
from dotenv import load_dotenv
load_dotenv()

print("✅ Entorno configurado correctamente.")

## Paso 1: Cargar Configuración

La configuración YAML define el dominio de investigación, los prompts para el agente, y los parámetros del modelo. Cada campo tiene un propósito específico:

- **`llm.base_url`**: Dónde corre el modelo de lenguaje. Usar un servidor local (vLLM) da control total sobre costos y privacidad.
- **`llm.model_name`**: El modelo específico. Qwen3.5, Llama, o cualquier modelo compatible con la API de OpenAI.
- **`target_sources`**: Cuántas fuentes web quiere explorar el agente. Más fuentes = más cobertura pero más tiempo.
- **`target_datasets`**: Cuántos datasets únicos espera encontrar. El pipeline se detiene al alcanzar este objetivo.
- **`dataset_goal`**: La descripción de lo que buscamos. Es el "prompt maestro" que guía todas las decisiones del agente.
- **`prompts.discovery_human`**: La instrucción que le dice al agente cómo generar sus consultas de búsqueda.

> **Nota de investigación**: El `dataset_goal` es el campo más importante. Una descripción precisa como *"leukocyte / white blood cell labeled datasets for segmentation, classification and identification"* produce resultados mucho mejores que una vaga como *"datasets de células"*. Cuanto más específico sea el dominio, mejor el agente puede generar consultas relevantes.

In [ ]:
config_path = "../configs/dataset_agent.yaml"

if not os.path.exists(config_path):
    print(f"Archivo de configuración no encontrado en {config_path}")
    print("Por favor créelo basado en configs/dataset_agent.yaml")
else:
    with open(config_path, 'r') as f:
        config = yaml.safe_load(f)
    
    print("=== Configuración Cargada ===")
    print(f"Modelo: {config['llm']['model_name']}")
    print(f"URL base: {config['llm']['base_url']}")
    print(f"Fuentes objetivo: {config.get('target_sources', 'N/A')}")
    print(f"Datasets objetivo: {config.get('target_datasets', 'N/A')}")
    print(f"Meta del dataset: {config.get('dataset_goal', 'N/A')}")
    print(f"Pasos máximos: {config.get('max_steps', 'N/A')}")
    print(f"Pasos de rescrape: {config.get('rescrape_steps', 'N/A')}")

## Paso 2: Esperar al Servidor LLM

El pipeline requiere un servidor de lenguaje grande (LLM) corriendo. Usamos **vLLM** porque:

- **Velocidad**: vLLM usa PagedAttention para inference altamente optimizado
- **Control**: Ejecutar localmente significa que no dependemos de APIs de terceros ni sus límites de rate
- **Costo**: Sin costos por token después de la inversión inicial en hardware
- **Privacidad**: Los datos de investigación nunca salen de tu infraestructura

La función `wait_for_server` verifica periódicamente si el modelo está cargado y listo para recibir solicitudes. Esto es crucial porque el modelo puede tardar minutos en cargarse después de iniciar vLLM.

In [ ]:
from scripts.wait_for_server import wait_for_server

base_url = config["llm"]["base_url"]
model_name = config["llm"]["model_name"]

print(f"Esperando servidor en {base_url}...")
ready = wait_for_server(base_url, model_name, timeout=300)

if ready:
    print("✅ ¡Servidor listo! El modelo está cargado.")
else:
    print("❌ Servidor no listo. Verifica que vLLM esté corriendo.")
    print("   Ejecuta: bash scripts/llama_serve.sh start")

## Paso 3: Construir y Ejecutar el Pipeline

Este es el núcleo del sistema. El pipeline funciona como un **grafo de estado dirigido** con 4 nodos principales:

### Flujo del grafo:

```
discover → crawl_extract → resolve_datasets → [router]
                                    ↓              ↓
                              summary ←── [loop/rescrape]
```

### Nodo 1: `discover` (Descubrimiento)
**¿Qué hace?** Genera consultas de búsqueda inteligentes usando un LLM, luego ejecuta cada consulta con Tavily Search para encontrar fuentes web relevantes.

**¿Por qué es necesario?** Los motores de búsqueda convencionales no entienden el contexto científico. Un LLM puede generar consultas como *"leukocyte segmentation dataset labeled images 2024"* que capturan la intención del investigador, no solo palabras clave.

**Ontología de Wikidata:** El agente consulta Wikidata para entender las jerarquías biológicas (ej: qué tipos de leucocitos existen, cómo se relacionan entre sí). Esto permite generar consultas que cubran todo el espacio semántico del dominio, no solo los términos más obvios.

### Nodo 2: `crawl_extract` (Extracción Agéntica)
**¿Qué hace?** Visita cada URL descubierta usando un agente autónomo que navega la página, extrae datasets, y estructura los metadatos (título, descripción, DOI, licencia, etc.).

**¿Por qué es necesario?** La mayoría de los datasets no están en formatos estandarizados. Un agente que puede leer páginas web arbitrarias, seguir enlaces, y extraer información estructuralmente es más versátil que un scraper rígido.

**Detección de duplicados:** Antes de guardar cada observación, el agente compara el embedding semántico con todos los existentes. Si la similitud cosine es > 0.92, se considera un duplicado y se descarta. Esto es mucho más robusto que comparar textos literalmente.

**Rescrape:** Después de los pasos máximos, el agente vuelve a las fuentes ya procesadas para extraer información adicional que pudo haber pasado por alto en la primera pasada.

### Nodo 3: `resolve_datasets` (Resolución de Datasets)
**¿Qué hace?** Compara cada observación con los datasets existentes usando similitud de embeddings. Si encuentra una coincidencia fuerte (> 0.75), un LLM verifica si deben fusionarse. Si no, crea un nuevo dataset.

**¿Por qué es necesario?** La deduplicación automática es crítica porque:
- Un dataset puede aparecer en 5 fuentes diferentes (GitHub, Zenodo, paper, repositorio institucional, blog)
- Cada fuente puede tener una descripción ligeramente diferente
- Sin deduplicación, tendríamos datasets duplicados que inflan los resultados y confunden al investigador

### Nodo 4: `summary` (Resumen)
**¿Qué hace?** Genera un resumen natural-language de todos los datasets encontrados.

**¿Por qué es necesario?** Un investigador necesita una vista de alto nivel antes de profundizar en detalles. El resumen responde: ¿cuántos datasets? ¿qué categorías? ¿cuáles son los más relevantes?

### Router (Enrutador)
**¿Qué hace?** Decide si volver a ejecutar el ciclo (loop), hacer rescrape, o terminar con el resumen.

**¿Por qué es necesario?** La investigación es iterativa. Si no encontramos suficientes fuentes en la primera pasada, volvemos a buscar. Si ya tenemos fuentes pero queremos más profundidad, hacemos rescrape.

In [ ]:
from dataset_agent.state_graph import build_graph
from dataset_agent.agents.state import init_state
from dataset_agent.db.db import save_db
from dataset_agent.utils.logging import get_logger

logger = get_logger(__name__)

# Inicializar estado desde la configuración
# El estado contiene: conexión a BD, modelo de embeddings, fase actual, contadores, etc.
state = init_state(config)

# Construir el grafo de estado con todos los nodos y conexiones
graph = build_graph()

# Ejecutar el pipeline
logger.info("Ejecutando el grafo de estado...")
final_state = graph.invoke(state)

# Guardar resultados en la base de datos
save_db(final_state["db"])
logger.info("Base de datos poblada con nuevos datasets.")

print("\n✅ Pipeline completado exitosamente.")
print(f"   Fase final: {final_state.get('phase', 'N/A')}")
print(f"   Resumen: {final_state.get('final_summary', 'N/A')[:200]}...")

## Paso 4: Visualizar Resultados

La visualización interactiva es una herramienta de investigación fundamental porque:

- **Revela patrones**: Puedes ver qué dominios (GitHub, arXiv, repositorios) producen más datasets
- **Muestra relaciones**: Las conexcciones entre búsquedas → fuentes → observaciones → datasets hacen visible el proceso de descubrimiento
- **Facilita la exploración**: Haz clic en cualquier nodo para ver sus metadatos completos
- **Identifica gaps**: Si un dominio específico (ej: repositorios de datos gubernamentales) tiene pocos nodos, sabes que necesitas buscar más ahí

### Colores del grafo:
| Color | Nodo | Significado |
|---|---|---|
| 🟣 Morado | Search | Consulta generada por el agente |
| 🟡 Amarillo | Source | URL descubierta (página web) |
| 🔵 Azul | Observation | Dataset identificado en una fuente |
| 🟢 Verde | Dataset | Dataset consolidado (después de deduplicación) |

In [ ]:
from scripts.visualize_connections import make_visualization

make_visualization()
print("✅ Visualización generada: data/dataset_connections.html")
print("\n💡 Haz clic en cualquier nodo para ver sus metadatos.")
print("   Doble clic en un nodo para abrir su URL en una nueva pestaña.")

## Paso 5: Explorar Resultados en la Base de Datos

La base de datos SQLite contiene 4 tablas principales. Entender su estructura es clave para hacer análisis personalizados:

| Tabla | Propósito | Campos clave |
|---|---|---|
| **searches** | Cada consulta generada por el agente | query, topic, created_at |
| **sources** | URLs descubiertas | url, webdomain, tavily_score, crawl_status |
| **observations** | Datasets identificados en fuentes | title, description, doi, confidence, status |
| **datasets** | Datasets consolidados (post-deduplicación) | title, description, doi, license_, keywords |

### ¿Por qué sqlite-vec?

`sqlite-vec` es una extensión de SQLite que permite almacenar y buscar embeddings vectoriales directamente dentro de la base de datos. Esto es poderoso porque:

1. **No necesitas un servicio separado**: No necesitas Pinecone, Weaviate, o Milvus para búsqueda semántica
2. **Todo está en un archivo**: La BD es un solo archivo `.db` que puedes copiar, compartir o versionar
3. **Búsqueda híbrida**: Puedes combinar búsqueda vectorial con filtros SQL tradicionales (ej: "datasets con DOI publicado después de 2023")
4. **Actualización incremental**: Los embeddings se recalculan y actualizan en cada ejecución del pipeline

> **Nota técnica**: Los embeddings se calculan como `embedding(title + "\n\n" + description)` usando el modelo de embeddings de OpenAI. La similitud cosine se usa para encontrar datasets similares.

In [ ]:
import sqlite3

db_path = "../data/datasets.db"

if os.path.exists(db_path):
    conn = sqlite3.connect(db_path)
    cur = conn.cursor()
    
    # Contar datasets encontrados
    cur.execute("SELECT COUNT(*) FROM datasets")
    total_datasets = cur.fetchone()[0]
    print(f"📊 Total de datasets encontrados: {total_datasets}")
    
    # Contar búsquedas realizadas
    cur.execute("SELECT COUNT(*) FROM searches")
    total_searches = cur.fetchone()[0]
    print(f"🔍 Total de búsquedas realizadas: {total_searches}")
    
    # Contar fuentes exploradas
    cur.execute("SELECT COUNT(*) FROM sources")
    total_sources = cur.fetchone()[0]
    print(f"🌐 Total de fuentes exploradas: {total_sources}")
    
    # Contar observaciones (datasets identificados antes de deduplicación)
    cur.execute("SELECT COUNT(*) FROM observations")
    total_observations = cur.fetchone()[0]
    print(f"📝 Total de observaciones: {total_observations}")
    
    # Mostrar datasets recientes
    print("\n📋 Últimos 10 datasets descubiertos:")
    print("-" * 80)
    cur.execute("SELECT id, title, license_, access_level FROM datasets ORDER BY id DESC LIMIT 10")
    for row in cur.fetchall():
        print(f"  #{row[0]}: {row[1]}")
        print(f"      Licencia: {row[2]} | Acceso: {row[3]}")
    
    # Mostrar dominios más frecuentes
    print("\n🏆 Dominios más frecuentes:")
    print("-" * 40)
    cur.execute("SELECT webdomain, COUNT(*) as cnt FROM sources WHERE webdomain IS NOT NULL GROUP BY webdomain ORDER BY cnt DESC LIMIT 10")
    for row in cur.fetchall():
        print(f"  {row[0]}: {row[1]} fuentes")
    
    # Mostrar consultas de búsqueda más productivas
    print("\n🔎 Consultas de búsqueda más productivas:")
    print("-" * 60)
    cur.execute("""
        SELECT s.query, COUNT(src.id) as source_count
        FROM searches s
        LEFT JOIN sources src ON s.id = src.search_id
        GROUP BY s.id
        ORDER BY source_count DESC
        LIMIT 10
    """)
    for row in cur.fetchall():
        print(f"  \"{row[0]}\" → {row[1]} fuentes")
    
    conn.close()
else:
    print(f"Base de datos no encontrada en {db_path}. Ejecuta el pipeline primero.")

## Paso 6: Análisis Avanzado

Ahora que tienes los datos, puedes hacer análisis más profundos. Aquí hay algunos ejemplos útiles para investigación:

### Analizar la calidad de los datasets
¿Cuántos datasets tienen DOI? ¿Cuántos tienen licencia abierta? ¿Cuál es la distribución de confianza?

In [ ]:
import sqlite3

if os.path.exists(db_path):
    conn = sqlite3.connect(db_path)
    cur = conn.cursor()
    
    # Calidad: ¿Cuántos tienen DOI?
    cur.execute("SELECT COUNT(*) FROM datasets WHERE doi IS NOT NULL")
    with_doi = cur.fetchone()[0]
    cur.execute("SELECT COUNT(*) FROM datasets")
    total = cur.fetchone()[0]
    print(f"📌 Datasets con DOI: {with_doi}/{total} ({100*with_doi/max(total,1):.1f}%)")
    
    # Calidad: ¿Cuántos tienen licencia?
    cur.execute("SELECT COUNT(*) FROM datasets WHERE license_ IS NOT NULL AND license_ != ''")
    with_license = cur.fetchone()[0]
    print(f"📜 Datasets con licencia: {with_license}/{total} ({100*with_license/max(total,1):.1f}%)")
    
    # Calidad: Distribución de confianza
    cur.execute("SELECT AVG(confidence), MIN(confidence), MAX(confidence) FROM observations")
    avg_c, min_c, max_c = cur.fetchone()
    print(f"\n📊 Confianza de observaciones:")
    print(f"   Promedio: {avg_c:.2f}")
    print(f"   Mínimo: {min_c:.2f}")
    print(f"   Máximo: {max_c:.2f}")
    
    # Datasets con alta confianza
    cur.execute("SELECT title, confidence FROM observations WHERE confidence > 0.8 ORDER BY confidence DESC LIMIT 5")
    print(f"\n🏅 Top 5 observaciones por confianza:")
    for row in cur.fetchall():
        print(f"   {row[0]} (confianza: {row[1]:.2f})")
    
    conn.close()
else:
    print("Base de datos no encontrada. Ejecuta el pipeline primero.")

### Analizar el rendimiento del agente
¿Cuántas fuentes por búsqueda? ¿Cuántas observaciones por fuente? ¿Cuántos datasets únicos por fuente?

In [ ]:
import sqlite3

if os.path.exists(db_path):
    conn = sqlite3.connect(db_path)
    cur = conn.cursor()
    
    # Eficiencia: fuentes por búsqueda
    cur.execute("""
        SELECT s.query, COUNT(src.id) as source_count
        FROM searches s
        LEFT JOIN sources src ON s.id = src.search_id
        GROUP BY s.id
        ORDER BY source_count DESC
    """)
    rows = cur.fetchall()
    if rows:
        avg_sources = sum(r[1] for r in rows) / len(rows)
        print(f"📈 Eficiencia de búsqueda:")
        print(f"   Promedio de fuentes por búsqueda: {avg_sources:.1f}")
        print(f"   Mejor búsqueda: \"{rows[0][0]}\" → {rows[0][1]} fuentes")
    
    # Eficiencia: observaciones por fuente
    cur.execute("""
        SELECT src.url, COUNT(obs.id) as obs_count
        FROM sources src
        LEFT JOIN observations obs ON src.id = obs.source_id
        GROUP BY src.id
        ORDER BY obs_count DESC
        LIMIT 5
    """)
    print(f"\n🏆 Top 5 fuentes por observaciones encontradas:")
    for row in cur.fetchall():
        print(f"   {row[0]} → {row[1]} observaciones")
    
    # Tasa de deduplicación
    cur.execute("SELECT COUNT(*) FROM observations")
    total_obs = cur.fetchone()[0]
    cur.execute("SELECT COUNT(*) FROM datasets")
    total_ds = cur.fetchone()[0]
    dedup_rate = 1 - (total_ds / max(total_obs, 1))
    print(f"\n🔄 Tasa de deduplicación: {dedup_rate:.1%}")
    print(f"   {total_obs} observaciones → {total_ds} datasets únicos")
    
    conn.close()
else:
    print("Base de datos no encontrada. Ejecuta el pipeline primero.")

## Conclusión

Este notebook demuestra un pipeline completo de investigación automatizada. Las herramientas utilizadas no son solo conveniencias técnicas — cada una resuelve un problema fundamental del descubrimiento científico:

1. **Tavily Search** supera las limitaciones de los motores de búsqueda convencionales para contenido científico
2. **Wikidata SPARQL** aporta estructura ontológica que guía búsquedas más inteligentes
3. **Embeddings semánticos** resuelven el problema de la duplicación semántica (misma cosa, palabras diferentes)
4. **LangGraph** orquesta la complejidad de múltiples agentes cooperativos
5. **sqlite-vec** proporciona búsqueda vectorial sin infraestructura adicional
6. **Visualización interactiva** transforma datos crudos en conocimiento accionable

El resultado es un sistema que puede descubrir, deduplicar y resumir datasets científicos de forma autónoma, liberando a los investigadores de la tediosa tarea de búsqueda manual.